# Membrane Segmentation: UNet vs UNet++ vs SimpleSegNet

**Dataset**: [ISBI 2012 EM Segmentation Challenge](http://brainiac2.mit.edu/isbi_challenge/) — electron microscopy images of neuronal structures (membrane / non-membrane)

**Goal**: Compare three segmentation architectures on a binary segmentation task:
- `SimpleUNet` — minimal encoder-decoder (baseline)
- `SimpleSegNet` — two-stage encoder-decoder
- `UNet++` — nested architecture with ResNeXt50 encoder (`segmentation_models_pytorch`)

**Bonus**: Compare the best model's predictions against manual threshold-based annotation.

---
| Model | Val Accuracy | Val Jaccard | Val F1 |
|-------|-------------|-------------|--------|
| SimpleUNet | ~0.86 | — | — |
| SimpleSegNet | ~0.85 | ~0.88 | ~0.94 |
| **UNet++** | **~0.91** | **~0.88** | **~0.94** |

## 1. Setup

In [ ]:
import os
import sys
import warnings
import torch
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import ReduceLROnPlateau
from segmentation_models_pytorch import UnetPlusPlus
from sklearn.metrics import accuracy_score, jaccard_score, fbeta_score
from tqdm.notebook import tqdm
from IPython.display import clear_output

sys.path.insert(0, os.path.abspath('.'))

from src import (
    GranulometryDataset,
    GranulometryDataModule,
    BCEDiceLoss,
    ModelCheckpoint,
    EarlyStopping,
    SimpleUNet,
    SimpleSegNet,
    show_first_batch,
    plot_metrics,
    visualize_predictions,
)
from src.dataset import create_dataframe

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Data

In [ ]:
IMAGE_DIR = 'data/membrane/train/image'
MASK_DIR  = 'data/membrane/train/mask'
IMG_HEIGHT = 512
IMG_WIDTH  = 512

df = create_dataframe(IMAGE_DIR, MASK_DIR)
print(f'Total samples: {len(df)}')
df.head()

In [ ]:
data_module = GranulometryDataModule(
    dataframe=df,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
    batch_size=2,
    num_workers=0,
)
data_module.setup()

train_loader = data_module.train_dataloader()
val_loader   = data_module.val_dataloader()
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
show_first_batch(train_loader)

## 3. Training Loop

In [ ]:
def calculate_metrics(outputs, masks, threshold=0.5):
    preds = (outputs > threshold).float().cpu().numpy().reshape(-1)
    targets = masks.cpu().numpy().reshape(-1)
    return (
        accuracy_score(targets, preds),
        jaccard_score(targets, preds),
        fbeta_score(targets, preds, beta=1),
    )


def train(
    model,
    model_name,
    train_loader,
    val_loader,
    device,
    num_epochs=50,
    lr=1e-2,
):
    model = model.to(device)
    optimizer  = torch.optim.Adam(model.parameters(), lr=lr)
    criterion  = BCEDiceLoss()
    scheduler  = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, threshold=0.003)
    checkpoint = ModelCheckpoint(save_dir=f'checkpoints/{model_name}', mode='min', max_models=2)
    stopper    = EarlyStopping(patience=7, mode='min', min_delta=0.003)

    metrics = {k: [] for k in ('loss', 'accuracy', 'jaccard_index', 'fbeta_score')}

    for epoch in range(num_epochs):
        # --- train ---
        model.train()
        t_loss = t_acc = t_jac = t_f1 = 0
        for imgs, msks in tqdm(train_loader, desc=f'[{model_name}] Epoch {epoch+1} train', leave=False):
            imgs = imgs.to(device, dtype=torch.float32)
            msks = msks.to(device, dtype=torch.float32)
            optimizer.zero_grad()
            out = torch.sigmoid(model(imgs))
            loss = criterion(out, msks)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
            acc, jac, f1 = calculate_metrics(out, msks)
            t_acc += acc; t_jac += jac; t_f1 += f1

        # --- val ---
        model.eval()
        v_loss = v_acc = v_jac = v_f1 = 0
        with torch.no_grad():
            for imgs, msks in tqdm(val_loader, desc=f'[{model_name}] Epoch {epoch+1} val', leave=False):
                imgs = imgs.to(device, dtype=torch.float32)
                msks = msks.to(device, dtype=torch.float32)
                out = torch.sigmoid(model(imgs))
                v_loss += criterion(out, msks).item()
                acc, jac, f1 = calculate_metrics(out, msks)
                v_acc += acc; v_jac += jac; v_f1 += f1

        n_tr, n_vl = len(train_loader), len(val_loader)
        t_loss /= n_tr; t_acc /= n_tr; t_jac /= n_tr; t_f1 /= n_tr
        v_loss /= n_vl; v_acc /= n_vl; v_jac /= n_vl; v_f1 /= n_vl

        for key, tr, vl in [
            ('loss', t_loss, v_loss),
            ('accuracy', t_acc, v_acc),
            ('jaccard_index', t_jac, v_jac),
            ('fbeta_score', t_f1, v_f1),
        ]:
            metrics[key].append({'train': tr, 'val': vl})

        scheduler.step(v_loss)
        checkpoint.step(epoch, v_loss, v_acc, model)

        clear_output(wait=True)
        print(f'[{model_name}] Epoch {epoch+1}/{num_epochs}')
        print(f'  Train  loss={t_loss:.4f}  acc={t_acc:.4f}  jaccard={t_jac:.4f}  f1={t_f1:.4f}')
        print(f'  Val    loss={v_loss:.4f}  acc={v_acc:.4f}  jaccard={v_jac:.4f}  f1={v_f1:.4f}')

        if stopper.step(v_loss):
            print('Early stopping triggered.')
            break

    return model, metrics

## 4. Experiment: SimpleUNet

In [ ]:
model_simpleunet, metrics_simpleunet = train(
    SimpleUNet(), 'SimpleUNet', train_loader, val_loader, device, num_epochs=50
)

In [ ]:
plot_metrics(metrics_simpleunet)

In [ ]:
visualize_predictions(model_simpleunet, val_loader, device, max_batches=2)

## 5. Experiment: SimpleSegNet

In [ ]:
model_segnet, metrics_segnet = train(
    SimpleSegNet(), 'SimpleSegNet', train_loader, val_loader, device, num_epochs=50
)

In [ ]:
plot_metrics(metrics_segnet)

In [ ]:
visualize_predictions(model_segnet, val_loader, device, max_batches=2)

## 6. Experiment: UNet++ (ResNeXt50)

In [ ]:
unetpp = UnetPlusPlus(
    encoder_name='resnext50_32x4d',
    encoder_weights='ssl',
    in_channels=1,
    classes=1,
)
model_unetpp, metrics_unetpp = train(
    unetpp, 'UNet++', train_loader, val_loader, device, num_epochs=50, lr=1e-2
)

In [ ]:
plot_metrics(metrics_unetpp)

In [ ]:
visualize_predictions(model_unetpp, val_loader, device, max_batches=2)

## 7. Model Comparison

In [ ]:
import pandas as pd

def best_val(metrics, key):
    vals = [m['val'] for m in metrics[key]]
    return max(vals) if key != 'loss' else min(vals)

rows = []
for name, m in [('SimpleUNet', metrics_simpleunet), ('SimpleSegNet', metrics_segnet), ('UNet++', metrics_unetpp)]:
    rows.append({
        'Model': name,
        'Best Val Loss': f"{best_val(m, 'loss'):.4f}",
        'Best Val Accuracy': f"{best_val(m, 'accuracy'):.4f}",
        'Best Val Jaccard': f"{best_val(m, 'jaccard_index'):.4f}",
        'Best Val F1': f"{best_val(m, 'fbeta_score'):.4f}",
    })

pd.DataFrame(rows)

## 8. Manual Annotation vs Model Prediction

Here we compare the UNet++ output against a manually created binary mask (threshold-based annotation with CLAHE preprocessing).

In [ ]:
import cv2
import numpy as np
from ipywidgets import interact

image_path = 'data/sample.png'  # replace with your image path
image_bgr = cv2.imread(image_path)

gray   = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
clahe  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enh    = clahe.apply(gray)
filt   = cv2.medianBlur(enh, 5)

def apply_threshold(threshold_value):
    _, mask = cv2.threshold(filt, threshold_value, 255, cv2.THRESH_BINARY)
    plt.figure(figsize=(12, 4))
    for i, (img, title) in enumerate([
        (cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), 'Original'),
        (filt, 'CLAHE + Median'),
        (mask, f'Threshold = {threshold_value}'),
    ]):
        plt.subplot(1, 3, i + 1)
        plt.imshow(img, cmap='gray' if img.ndim == 2 else None)
        plt.title(title); plt.axis('off')
    plt.tight_layout(); plt.show()

interact(apply_threshold, threshold_value=(0, 255, 1))

In [ ]:
import torchvision.transforms as T

manual_mask_path = 'data/mask_manual.png'  # your drawn / thresholded mask

def preprocess(path):
    img = Image.open(path).convert('L')
    tf  = T.Compose([T.Resize((IMG_HEIGHT, IMG_WIDTH)), T.ToTensor()])
    return tf(img).unsqueeze(0)

tensor = preprocess(image_path).to(device)
model_unetpp.eval()
with torch.no_grad():
    pred = torch.argmax(torch.softmax(model_unetpp(tensor), dim=1), dim=1)

true_mask = Image.open(manual_mask_path).convert('L').resize((IMG_HEIGHT, IMG_WIDTH))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(Image.open(image_path).convert('L'), cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(pred.cpu().squeeze(), cmap='gray');                 axes[1].set_title('Model Prediction'); axes[1].axis('off')
axes[2].imshow(true_mask, cmap='gray');                            axes[2].set_title('Manual Mask'); axes[2].axis('off')
plt.tight_layout(); plt.show()